In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE         = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
HORAS_IN     = BASE / "data/processed/horas_por_setor_2023.csv"
PIB_IN       = BASE / "data/processed/pib_setorial_2012_2023.csv"
ARQUIVO_OUT  = BASE / "data/processed/produtividade_por_hora_2023.csv"
DIR_FIGS     = BASE / "outputs/figures"

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})
print("Imports OK")

In [ ]:
df_horas = pd.read_csv(HORAS_IN, index_col=0)
df_pib   = pd.read_csv(PIB_IN)

print("── Horas por setor (PNAD 2023) ─────────────────────────")
print(df_horas[["n", "media", "mediana"]].to_string())

print("\n── PIB setorial 2023 (R$ milhões) ───────────────────────")
pib_2023 = df_pib[df_pib["ano"] == 2023].squeeze()
print(pib_2023.to_string())

In [ ]:
# Os grupamentos da PNAD (VD4010) não são idênticos aos da CNT
# Decisão: fazer correspondência manual baseada nas definições do IBGE
# Estrutura:
# setor_pnad: nome usado no dataset de horas
# col_cnt: coluna correspondente no dataset do PIB
# nota: onde há divergência ou agregação

correspondencia = [
    {
        "setor_pnad":  "Agropecuária",
        "col_cnt":     "agropecuaria",
        "nota":        "Correspondência direta"
    },
    {
        "setor_pnad":  "Indústria geral",
        "col_cnt":     "ind_transformacao",
        "nota":        "PNAD agrega toda indústria; CNT separa. Usamos transformação como proxy principal (maior parcela)"
    },
    {
        "setor_pnad":  "Construção",
        "col_cnt":     "construcao",
        "nota":        "Correspondência direta"
    },
    {
        "setor_pnad":  "Comércio e rep.",
        "col_cnt":     "comercio",
        "nota":        "Correspondência direta"
    },
    {
        "setor_pnad":  "Transp. e armaz.",
        "col_cnt":     "transporte",
        "nota":        "Correspondência direta"
    },
    {
        "setor_pnad":  "Alojamento e alim.",
        "col_cnt":     "outros_servicos",
        "nota":        "CNT não separa alojamento — incluído em outros serviços. Proxy imperfeita."
    },
    {
        "setor_pnad":  "Inf. e serv. prof.",
        "col_cnt":     "info_comunicacao",
        "nota":        "PNAD agrega info + serv. prof.; CNT separa. Usamos info_comunicacao como proxy."
    },
    {
        "setor_pnad":  "Adm. públ. e educação",
        "col_cnt":     "adm_publica_saude_educ",
        "nota":        "CNT agrega adm. pública + saúde + educação públicas. Correspondência ampla."
    },
    {
        "setor_pnad":  "Saúde",
        "col_cnt":     "adm_publica_saude_educ",
        "nota":        "Saúde pública está agregada com adm. pública na CNT. Proxy compartilhada — limitação importante."
    },
    {
        "setor_pnad":  "Serv. domésticos",
        "col_cnt":     "outros_servicos",
        "nota":        "Serv. domésticos têm baixo VAB formal na CNT. Proxy muito imperfeita."
    },
]

df_corr = pd.DataFrame(correspondencia)
print("\n── Tabela de correspondência ────────────────────────────")
print(df_corr[["setor_pnad", "col_cnt", "nota"]].to_string(index=False))

In [ ]:
# Fórmula:
#   produtividade_hora = VAB_setor (R$ milhões) / total_horas_setor
#   total_horas_setor  = n_ocupados × horas_media_semanal × 52 semanas
#
# Unidade: R$ por hora trabalhada

resultados = []

for _, row in df_corr.iterrows():
    setor  = row["setor_pnad"]
    col    = row["col_cnt"]

    if setor not in df_horas.index:
        continue

    n_ocupados   = df_horas.loc[setor, "n"]
    horas_media  = df_horas.loc[setor, "media"]
    vab_milhoes  = pib_2023[col]

    total_horas_ano = n_ocupados * horas_media * 52 / 1e6

    # Produtividade: R$ por hora (VAB em R$ / horas totais)
    # VAB está em R$ milhões -> multiplicar por 1e6
    produt_hora = (vab_milhoes * 1e6) / (total_horas_ano * 1e6)

    resultados.append({
        "setor":             setor,
        "n_ocupados_pnad":   int(n_ocupados),
        "horas_media":       round(horas_media, 1),
        "total_horas_ano_M": round(total_horas_ano, 2),
        "vab_r$_bi":         round(vab_milhoes / 1e3, 1),
        "produt_r$_hora":    round(produt_hora, 2),
        "col_cnt":           col,
        "nota":              row["nota"],
    })

df_prod = pd.DataFrame(resultados).sort_values("produt_r$_hora", ascending=False)

print("\n── Produtividade por hora trabalhada — Brasil 2023 ──────")
print(df_prod[["setor", "horas_media", "vab_r$_bi", "produt_r$_hora"]].to_string(index=False))

df_prod.to_csv(ARQUIVO_OUT, index=False)
print(f"\n✓ Salvo: {ARQUIVO_OUT}")

In [ ]:
df_g = df_prod[~df_prod["setor"].isin(["Saúde", "Serv. domésticos", "Alojamento e alim."])].copy()
df_g = df_g.sort_values("produt_r$_hora", ascending=True)

media_geral = df_prod["produt_r$_hora"].mean()

fig, ax = plt.subplots(figsize=(11, 6))

cores = [
    "#27AE60" if v > media_geral * 1.2
    else "#E84545" if v < media_geral * 0.8
    else "#2C6FBF"
    for v in df_g["produt_r$_hora"]
]

bars = ax.barh(df_g["setor"], df_g["produt_r$_hora"], color=cores, height=0.6)
ax.axvline(media_geral, color="gray", linewidth=1.5, linestyle="--",
           label=f"Média geral: R$ {media_geral:.0f}/h")

for bar, val in zip(bars, df_g["produt_r$_hora"]):
    ax.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height() / 2,
        f"R$ {val:.0f}/h", va="center", fontsize=9
    )

ax.set_xlabel("Produtividade estimada (R$/hora trabalhada)", fontsize=11)
ax.set_title(
    "Proxy de produtividade por hora trabalhada — Brasil 2023\n"
    "VAB setorial (CNT/IBGE) ÷ horas totais estimadas (PNAD Contínua)",
    fontsize=12, loc="left"
)
ax.set_xlim(0, df_g["produt_r$_hora"].max() * 1.25)
ax.legend(frameon=False, fontsize=10)

fig.text(
    0.01, -0.03,
    "⚠ Proxy: numerador (VAB) e denominador (horas PNAD) têm cobertura diferente. "
    "Usar apenas como indicador relativo entre setores.",
    fontsize=7.5, color="gray", transform=fig.transFigure
)

plt.tight_layout()
plt.savefig(DIR_FIGS / "09_produtividade_por_hora_setor.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 9 salvo.")

In [ ]:
df_scatter = df_prod[~df_prod["setor"].isin(["Saúde", "Serv. domésticos"])].copy()

fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(
    df_scatter["horas_media"],
    df_scatter["produt_r$_hora"],
    s=df_scatter["n_ocupados_pnad"] / 3000,
    color="#2C6FBF", alpha=0.7, edgecolors="white", linewidth=0.8
)

for _, row in df_scatter.iterrows():
    ax.annotate(
        row["setor"],
        (row["horas_media"], row["produt_r$_hora"]),
        textcoords="offset points", xytext=(6, 3),
        fontsize=8.5, color="black"
    )

ax.axvline(40, color="#2C6FBF", linewidth=1.2, linestyle="--", alpha=0.6, label="40h (proposta)")
ax.axvline(44, color="#F5A623", linewidth=1.2, linestyle="--", alpha=0.6, label="44h (CLT atual)")

# Linha de tendência
z = np.polyfit(df_scatter["horas_media"], df_scatter["produt_r$_hora"], 1)
p = np.poly1d(z)
x_line = np.linspace(df_scatter["horas_media"].min(), df_scatter["horas_media"].max(), 100)
ax.plot(x_line, p(x_line), color="gray", linewidth=1, linestyle=":", label="Tendência linear")

ax.set_xlabel("Jornada média habitual semanal (horas)", fontsize=11)
ax.set_ylabel("Produtividade estimada (R$/hora)", fontsize=11)
ax.set_title(
    "Jornada média × produtividade por hora — setores — Brasil 2023\n"
    "Tamanho do ponto proporcional ao nº de ocupados · PNAD + CNT/IBGE",
    fontsize=12, loc="left"
)
ax.legend(frameon=False, fontsize=10)

fig.text(
    0.01, -0.03,
    "⚠ Proxy com limitações de cobertura. Correlação não implica causalidade.",
    fontsize=7.5, color="gray", transform=fig.transFigure
)

plt.tight_layout()
plt.savefig(DIR_FIGS / "10_scatter_horas_produtividade.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 10 salvo.")